<a href="https://colab.research.google.com/github/mohanasankariS/readai-clone/blob/main/intern_notesgenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
# ==========================================
# Cell 1 - Install Required Libraries
# ==========================================

!apt-get update -qq
!apt-get install -y -qq ffmpeg

!pip install -q \
gradio \
yt-dlp \
opencv-python \
python-docx \
reportlab \
groq \
google-genai \
markdown \
pillow \
tqdm

print("✅ All libraries installed successfully.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ All libraries installed successfully.


In [71]:
# ==========================================
# Cell 2 - Import Libraries
# ==========================================

import os
import glob
import shutil
import subprocess
import cv2
import yt_dlp
import gradio as gr

from pathlib import Path
from datetime import datetime
from tqdm import tqdm

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [72]:
# ==========================================
# Cell 3 - Mount Google Drive & Check cookies
# ==========================================

from google.colab import drive

drive.mount('/content/drive')

COOKIE_PATH = "/content/drive/MyDrive/cookies.txt"

if os.path.exists(COOKIE_PATH):
    print("✅ cookies.txt found")
    print(COOKIE_PATH)
else:
    print("❌ cookies.txt NOT found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ cookies.txt found
/content/drive/MyDrive/cookies.txt


In [73]:
# ==========================================
# Cell 4 - Create Project Folders
# ==========================================

folders = [
    "downloads",
    "audio",
    "audio_chunks",
    "slides",
    "transcript",
    "notes",
    "pdf",
    "docx"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created.")

✅ Project folders created.


In [74]:
# ==========================================
# Cell 5 - Download YouTube Video
# ==========================================

import yt_dlp
import glob
import shutil

def download_youtube_video(url):

    # Remove old files
    for file in glob.glob("downloads/lecture_input*"):
        os.remove(file)

    output_template = "downloads/lecture_input.%(ext)s"

    ydl_opts = {

        "format": "bestvideo+bestaudio/best",

        "merge_output_format": "mp4",

        "outtmpl": output_template,

        "cookiefile": COOKIE_PATH,

        "noplaylist": True,

        "overwrites": True,

        "quiet": False
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    files = glob.glob("downloads/lecture_input.*")

    if len(files) == 0:
        raise Exception("❌ Video download failed.")

    downloaded_file = files[0]

    final_video = "downloads/lecture_input.mp4"

    if downloaded_file != final_video:

        if os.path.exists(final_video):
            os.remove(final_video)

        shutil.move(downloaded_file, final_video)

    print("✅ Download Complete")

    return final_video

In [75]:
# ==========================================
# Cell 6 - Save Uploaded Video
# ==========================================

def save_uploaded_video(video_path):

    destination = "downloads/lecture_input.mp4"

    if os.path.exists(destination):
        os.remove(destination)

    shutil.copy(video_path, destination)

    print("✅ Uploaded video saved.")

    return destination

In [76]:
# ==========================================
# Cell 7 - Extract Audio
# ==========================================

import subprocess

def extract_audio(video_path):

    audio_path = "audio/lecture_audio.wav"

    if os.path.exists(audio_path):
        os.remove(audio_path)

    command = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        audio_path
    ]

    subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if not os.path.exists(audio_path):
        raise Exception("❌ Audio extraction failed.")

    print("✅ Audio extracted.")

    return audio_path

In [77]:
# ==========================================
# Cell 8 - Split Audio into Chunks
# ==========================================

import os
import glob
import subprocess

def split_audio(audio_path):

    # Remove previous chunks
    for file in glob.glob("audio_chunks/*.wav"):
        os.remove(file)

    output_pattern = "audio_chunks/chunk_%03d.wav"

    command = [
        "ffmpeg",
        "-i", audio_path,
        "-f", "segment",
        "-segment_time", "600",
        "-c", "copy",
        output_pattern
    ]

    subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    chunk_files = sorted(glob.glob("audio_chunks/*.wav"))

    print(f"✅ {len(chunk_files)} audio chunks created.")

    return chunk_files

In [78]:
# ==========================================
# Cell 9 - Extract Slides
# ==========================================

import cv2

def extract_slides(video_path):

    for file in glob.glob("slides/*.jpg"):
        os.remove(file)

    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS))

    if fps <= 0:
        fps = 30

    frame_interval = fps * 60

    frame_number = 0
    slide_number = 1

    slide_list = []

    while True:

        success, frame = cap.read()

        if not success:
            break

        if frame_number % frame_interval == 0:

            filename = f"slides/slide_{slide_number}.jpg"

            cv2.imwrite(filename, frame)

            slide_list.append(filename)

            slide_number += 1

        frame_number += 1

    cap.release()

    print(f"✅ {len(slide_list)} slides extracted.")

    return slide_list

In [79]:
# ==========================================
# Cell 10 - Load Groq API
# ==========================================

from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise Exception("❌ GROQ_API_KEY not found in Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq API connected successfully.")

✅ Groq API connected successfully.


In [80]:
# ==========================================
# Cell 11 - Transcribe Audio Chunks
# ==========================================

def transcribe_audio_chunks(chunk_files):

    full_transcript = ""

    for i, chunk in enumerate(chunk_files):

        print(f"🎤 Transcribing Chunk {i+1}/{len(chunk_files)}")

        with open(chunk, "rb") as audio_file:

            response = client.audio.transcriptions.create(

                file=(os.path.basename(chunk), audio_file),

                model="whisper-large-v3-turbo",

                response_format="text"

            )

        full_transcript += response + "\n\n"

    transcript_path = "transcript/transcript.txt"

    with open(transcript_path, "w", encoding="utf-8") as f:

        f.write(full_transcript)

    print("✅ Complete transcript created.")

    return full_transcript

In [90]:
# ==========================================
# Updated Cell 12 - Chunked AI Notes Generation
# ==========================================

import time

def split_text_into_chunks(text, max_words=2000):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)
    return chunks

def generate_notes(transcript):
    chunks = split_text_into_chunks(transcript, max_words=2000)
    print(f"🧩 Processing transcript in {len(chunks)} chunks...")

    chunk_summaries = []

    # Process each chunk individually to stay within the 6,000 TPM limit
    for i, chunk in enumerate(chunks):
        print(f"📝 Summarizing section {i+1}/{len(chunks)}...")

        prompt = f"""
Summarize this portion of a lecture into clear key points, main concepts, definitions, and examples:

{chunk}
"""
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        chunk_summaries.append(response.choices[0].message.content)

        # Brief pause between calls to avoid hitting rate limits
        time.sleep(2)

    combined_summary = "\n\n".join(chunk_summaries)

    # Final pass to assemble standard lecture notes
    final_prompt = f"""
You are an expert university professor.

Below are structured key points from a full lecture. Compile them into a complete set of study notes in English.

Output format:
# Title
## Introduction
## Concepts
## Definitions
## Examples
## Important Points
## Summary
## Exam Questions
### 2 Marks
### 5 Marks
### 10 Marks

Combined Points:
{combined_summary}
"""

    print("🚀 Generating final structured notes...")
    final_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": final_prompt}]
    )

    notes = final_response.choices[0].message.content

    notes_path = "notes/lecture_notes.md"
    with open(notes_path, "w", encoding="utf-8") as f:
        f.write(notes)

    print("✅ Notes Generated Successfully!")
    return notes

In [95]:
# ==========================================
# Updated Cell 13 - Proper PDF Formatting
# ==========================================

import re
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

def markdown_to_reportlab_html(text):
    # Convert bold **text** to <b>text</b>
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
    # Convert italic *text* to <i>text</i>
    text = re.sub(r'\*(.*?)\*', r'<i>\1</i>', text)
    return text

def create_pdf(notes):
    pdf_path = "pdf/Lecture_Notes.pdf"
    doc = SimpleDocTemplate(
        pdf_path,
        rightMargin=36, leftMargin=36,
        topMargin=36, bottomMargin=36
    )

    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle(
        'DocTitle', parent=styles['Heading1'],
        fontSize=20, leading=24, textColor=colors.HexColor("#1A2B4C"),
        spaceAfter=12
    )
    h2_style = ParagraphStyle(
        'DocH2', parent=styles['Heading2'],
        fontSize=14, leading=18, textColor=colors.HexColor("#2B547E"),
        spaceBefore=10, spaceAfter=6
    )
    body_style = ParagraphStyle(
        'DocBody', parent=styles['BodyText'],
        fontSize=10, leading=14, spaceAfter=4
    )
    bullet_style = ParagraphStyle(
        'DocBullet', parent=styles['BodyText'],
        fontSize=10, leading=14, leftIndent=15, spaceAfter=3
    )

    story = []

    for line in notes.split("\n"):
        line = line.strip()
        if not line:
            continue

        # Parse Markdown Headers
        if line.startswith("# "):
            clean_line = markdown_to_reportlab_html(line[2:])
            story.append(Paragraph(clean_line, title_style))
        elif line.startswith("## ") or line.startswith("### "):
            clean_line = markdown_to_reportlab_html(line.lstrip("#").strip())
            story.append(Paragraph(clean_line, h2_style))
        # Parse Bullet Points
        elif line.startswith("* ") or line.startswith("- "):
            clean_line = markdown_to_reportlab_html(line[2:])
            story.append(Paragraph(f"• {clean_line}", bullet_style))
        else:
            clean_line = markdown_to_reportlab_html(line)
            story.append(Paragraph(clean_line, body_style))

    doc.build(story)
    print("✅ Formatted PDF Generated")
    return pdf_path

In [96]:
# ==========================================
# Updated Cell 14 - Proper DOCX Formatting
# ==========================================

import re
from docx import Document

def add_formatted_paragraph(doc, text, style=None):
    p = doc.add_paragraph(style=style)
    # Split text by bold markers **
    parts = re.split(r'(\*\*.*?\*\*)', text)
    for part in parts:
        if part.startswith('**') and part.endswith('**'):
            p.add_run(part[2:-2]).bold = True
        else:
            p.add_run(part)

def create_docx(notes):
    doc = Document()

    for line in notes.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("# "):
            doc.add_heading(line[2:], level=1)
        elif line.startswith("## "):
            doc.add_heading(line[3:], level=2)
        elif line.startswith("### "):
            doc.add_heading(line[4:], level=3)
        elif line.startswith("* ") or line.startswith("- "):
            add_formatted_paragraph(doc, line[2:], style='List Bullet')
        else:
            add_formatted_paragraph(doc, line)

    docx_path = "docx/Lecture_Notes.docx"
    doc.save(docx_path)
    print("✅ Formatted DOCX Generated")
    return docx_path

In [97]:
# ==========================================
# Cell 15 - Main Process
# ==========================================

def process_lecture(video_file, youtube_url):

    try:

        # Download or Upload
        if youtube_url.strip():

            status="Downloading YouTube Video..."

            video_path=download_youtube_video(youtube_url)

        elif video_file:

            status="Saving Uploaded Video..."

            video_path=save_uploaded_video(video_file)

        else:

            return "Upload video or enter YouTube URL.",None,None

        # Audio
        audio=extract_audio(video_path)

        # Split Audio
        chunks=split_audio(audio)

        # Slides
        slides=extract_slides(video_path)

        # Transcription
        transcript=transcribe_audio_chunks(chunks)

        # AI Notes
        notes=generate_notes(transcript)

        # PDF
        pdf=create_pdf(notes)

        # DOCX
        doc=create_docx(notes)

        status=f"""

✅ Video Processed

🎤 Audio Extracted

✂ {len(chunks)} Audio Chunks

🖼 {len(slides)} Slides Extracted

📝 Notes Generated

📄 PDF Ready

📘 DOCX Ready

"""

        return status,pdf,doc

    except Exception as e:

        return str(e),None,None

In [98]:
# ==========================================
# Cell 16 - Final Gradio UI
# ==========================================

import gradio as gr

with gr.Blocks(
    title="AI Lecture Notes Generator",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎓 AI Lecture Notes Generator

    ### Features

    ✅ Upload Lecture Video

    ✅ YouTube Video Support

    ✅ Supports Long Lecture Videos (1+ Hour)

    ✅ Automatic Audio Extraction

    ✅ Audio Chunking

    ✅ AI Transcription

    ✅ English Notes Generation

    ✅ Professional PDF

    ✅ Word Document

    """)

    with gr.Row():

        with gr.Column():

            video_input = gr.Video(
                label="📹 Upload Lecture Video"
            )

            youtube_input = gr.Textbox(
                label="🌐 YouTube URL",
                placeholder="https://youtube.com/..."
            )

            generate_button = gr.Button(
                "🚀 Generate Notes",
                variant="primary"
            )

        with gr.Column():

            status_output = gr.Textbox(
                label="Processing Status",
                lines=15
            )

            pdf_output = gr.File(
                label="📄 Download PDF"
            )

            docx_output = gr.File(
                label="📘 Download DOCX"
            )

    generate_button.click(
        fn=process_lecture,
        inputs=[
            video_input,
            youtube_input
        ],
        outputs=[
            status_output,
            pdf_output,
            docx_output
        ]
    )

demo.launch(
    debug=True,
    share=True
)

  with gr.Blocks(



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aa2b133fd354b03cb4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[youtube] Extracting URL: https://youtu.be/YBMRrerCEqE?si=1cAuyOTpB9ElN0qO
[youtube] YBMRrerCEqE: Downloading webpage


[youtube] YBMRrerCEqE: Downloading android vr player API JSON
[info] YBMRrerCEqE: Downloading 1 format(s): 398+251
[download] Destination: downloads/lecture_input.f398.mp4
[download] 100% of   71.20MiB in 00:00:11 at 5.96MiB/s   
[download] Destination: downloads/lecture_input.f251.webm
[download] 100% of   37.80MiB in 00:00:04 at 8.82MiB/s   
[Merger] Merging formats into "downloads/lecture_input.mp4"
Deleting original file downloads/lecture_input.f251.webm (pass -k to keep)
Deleting original file downloads/lecture_input.f398.mp4 (pass -k to keep)
✅ Download Complete
✅ Audio extracted.
✅ 5 audio chunks created.
✅ 0 slides extracted.
🎤 Transcribing Chunk 1/5
🎤 Transcribing Chunk 2/5
🎤 Transcribing Chunk 3/5
🎤 Transcribing Chunk 4/5
🎤 Transcribing Chunk 5/5
✅ Complete transcript created.
🧩 Processing transcript in 4 chunks...
📝 Summarizing section 1/4...
📝 Summarizing section 2/4...
📝 Summarizing section 3/4...
📝 Summarizing section 4/4...
🚀 Generating final structured notes...
✅ Notes 